<a href="https://colab.research.google.com/github/marcovirulucas/econ_apis/blob/main/united_states/FRED_01_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Federal Reserve Economic Data (FRED) API with Python
## Part 1 - Exploration
## Releases, Series and Metadata
------
*September 22, 2026*\
\
The FRED API documentation can be found [here](https://fred.stlouisfed.org/docs/api/fred/) (version 1) and [here](https://fred.stlouisfed.org/docs/api/fred/v2/index.html) (version 2). A free API key can be requested [here](https://fredaccount.stlouisfed.org/apikeys).\
\
This script finds the releases and series available in FRED, and the codes (series IDs) needed to request the Federal Reserve's economic projections, oil prices and PCE inflation.

In [1]:
import requests
import pandas as pd

### API key
Every request to FRED must include an API key. In Google Colab, save it in *Secrets* (key icon on the left panel) with the name `FRED_API_KEY`. Outside Colab, paste it directly in the cell below.

In [2]:
# Load the API key (never share it or upload it to GitHub)
try:
    from google.colab import userdata
    API_KEY = userdata.get('FRED_API_KEY')
except ImportError:
    import keys
    API_KEY = '{}'.format(keys.fred_key)

### Connecting to the server
All metadata endpoints share the same base URL. Each request needs the parameters `api_key` and `file_type=json`, so we write a small function that adds them and returns the response as a dictionary.

In [3]:
base = 'https://api.stlouisfed.org/fred/'

def fred_get(endpoint, **params):
    """Send a request to a FRED endpoint and return the JSON response as a dictionary."""
    params.update({'api_key': API_KEY, 'file_type': 'json'})
    resp = requests.get(base + endpoint, params=params)
    resp.raise_for_status()
    return resp.json()

### Searching for the release
FRED organizes its data in *releases* (publications), such as the Consumer Price Index, the Employment Situation or the FOMC Summary of Economic Projections. A release plays the role of a dataset.

In [4]:
# Get all releases
releases = pd.DataFrame(fred_get('releases', limit=1000)['releases'])

# Search for releases containing "Projections"
releases.loc[releases['name'].str.contains('Projections'), ['id', 'name', 'link']]

,id,name,link
181,326,Summary of Economic Projections,http://www.federalreserve.gov/monetarypolicy/f...


The search result shows the release ID (326) and its name.

In [5]:
# Obtain the metadata for the Summary of Economic Projections release
fred_get('release', release_id=326)['releases'][0]

{'id': 326,
 'realtime_start': '2026-09-22',
 'realtime_end': '2026-09-22',
 'name': 'Summary of Economic Projections',
 'press_release': False,
 'link': 'http://www.federalreserve.gov/monetarypolicy/fomccalendars.htm'}

### Getting the series of the release
A release is made up of many series. Each series has an ID (e.g., `PCECTPIMD`) and metadata: title, frequency, units and seasonal adjustment. *Note: FRED names this key `seriess` (with a double s).*

In [6]:
# List all series that belong to the release
sep = pd.DataFrame(fred_get('release/series', release_id=326)['seriess'])
sep = sep[['id', 'title', 'frequency', 'units', 'observation_start', 'observation_end', 'last_updated']]
print(len(sep), 'series')
sep.head(10)

63 series


,id,title,frequency,units,observation_start,observation_end,last_updated
0,FEDTARCTH,FOMC Summary of Economic Projections for the F...,Annual,Percent,2026-01-01,2029-01-01,2026-09-16 14:09:30-05
1,FEDTARCTHLR,Longer Run FOMC Summary of Economic Projection...,Not Applicable,Percent,2015-06-17,2026-09-16,2026-09-16 14:09:30-05
2,FEDTARCTL,FOMC Summary of Economic Projections for the F...,Annual,Percent,2026-01-01,2029-01-01,2026-09-16 14:09:30-05
3,FEDTARCTLLR,Longer Run FOMC Summary of Economic Projection...,Not Applicable,Percent,2015-06-17,2026-09-16,2026-09-16 14:09:30-05
4,FEDTARCTM,FOMC Summary of Economic Projections for the F...,Annual,Percent,2026-01-01,2029-01-01,2026-09-16 14:09:29-05
5,FEDTARCTMLR,Longer Run FOMC Summary of Economic Projection...,Not Applicable,Percent,2015-06-17,2026-09-16,2026-09-16 14:09:29-05
6,FEDTARMD,FOMC Summary of Economic Projections for the F...,Annual,Percent,2026-01-01,2029-01-01,2026-09-16 14:09:29-05
7,FEDTARMDLR,Longer Run FOMC Summary of Economic Projection...,Not Applicable,Percent,2012-01-25,2026-09-16,2026-09-16 14:09:29-05
8,FEDTARRH,FOMC Summary of Economic Projections for the F...,Annual,Percent,2026-01-01,2029-01-01,2026-09-16 14:09:30-05
9,FEDTARRHLR,Longer Run FOMC Summary of Economic Projection...,Not Applicable,Percent,2015-06-17,2026-09-16,2026-09-16 14:09:28-05


### Understanding the structure of the series IDs
The series IDs of this release are built like the dimensions of an SDMX dataset: **VARIABLE + STATISTIC + HORIZON**. For example, `PCECTPI` + `MD` = median projection of PCE inflation, and `FEDTAR` + `MD` + `LR` = median longer-run projection of the federal funds rate.

In [7]:
# Split each ID into its three components
pattern = r'^(GDPC1|UNRATE|PCECTPI|JCXFE|FEDTAR)(MD|CTH|CTL|CTM|RH|RL|RM)(LR)?$'
parts = sep['id'].str.extract(pattern)
parts.columns = ['VARIABLE', 'STATISTIC', 'HORIZON']
parts['HORIZON'] = parts['HORIZON'].fillna('Annual')
sep = pd.concat([sep, parts], axis=1)

# Number of series for each combination of variable and statistic
pd.crosstab(sep['VARIABLE'], sep['STATISTIC'])

STATISTIC,CTH,CTL,CTM,MD,RH,RL,RM
VARIABLE,,,,,,,
FEDTAR,2,2,2,2,2,2,2
GDPC1,2,2,2,2,2,2,2
JCXFE,1,1,1,1,1,1,1
PCECTPI,2,2,2,2,2,2,2
UNRATE,2,2,2,2,2,2,2


### Getting the codes for each dimension
Here we look up the variables. The median annual projections show one series per variable.

In [8]:
# Codes of the VARIABLE dimension
sep.loc[sep['STATISTIC'].eq('MD') & sep['HORIZON'].eq('Annual'), ['id', 'title', 'units']]

,id,title,units
6,FEDTARMD,FOMC Summary of Economic Projections for the F...,Percent
20,GDPC1MD,FOMC Summary of Economic Projections for the G...,Fourth Quarter to Fourth Quarter Percent Change
31,JCXFEMD,FOMC Summary of Economic Projections for the P...,Fourth Quarter to Fourth Quarter Percent Change
41,PCECTPIMD,FOMC Summary of Economic Projections for the P...,Fourth Quarter to Fourth Quarter Percent Change
55,UNRATEMD,FOMC Summary of Economic Projections for the C...,Percent


Here we look up the statistics. The code "MD" (median) will be selected, as in the table of Summary Projections.

In [9]:
# Codes of the STATISTIC dimension (read from the titles of the PCE inflation series)
sep.loc[sep['VARIABLE'].eq('PCECTPI'), ['id', 'STATISTIC', 'HORIZON', 'title']]

,id,STATISTIC,HORIZON,title
35,PCECTPICTH,CTH,Annual,FOMC Summary of Economic Projections for the P...
36,PCECTPICTHLR,CTH,LR,Longer Run FOMC Summary of Economic Projection...
37,PCECTPICTL,CTL,Annual,FOMC Summary of Economic Projections for the P...
38,PCECTPICTLLR,CTL,LR,Longer Run FOMC Summary of Economic Projection...
39,PCECTPICTM,CTM,Annual,FOMC Summary of Economic Projections for the P...
40,PCECTPICTMLR,CTM,LR,Longer Run FOMC Summary of Economic Projection...
41,PCECTPIMD,MD,Annual,FOMC Summary of Economic Projections for the P...
42,PCECTPIMDLR,MD,LR,Longer Run FOMC Summary of Economic Projection...
43,PCECTPIRH,RH,Annual,FOMC Summary of Economic Projections for the P...
44,PCECTPIRHLR,RH,LR,Longer Run FOMC Summary of Economic Projection...


We can also search within the titles. In this case, we find the codes of core PCE inflation ("less Food and Energy"):

In [10]:
# Search an specific series
sep.loc[sep['title'].str.contains('less Food and Energy'), ['id', 'title']]

,id,title
28,JCXFECTH,FOMC Summary of Economic Projections for the P...
29,JCXFECTL,FOMC Summary of Economic Projections for the P...
30,JCXFECTM,FOMC Summary of Economic Projections for the P...
31,JCXFEMD,FOMC Summary of Economic Projections for the P...
32,JCXFERH,FOMC Summary of Economic Projections for the P...
33,JCXFERL,FOMC Summary of Economic Projections for the P...
34,JCXFERM,FOMC Summary of Economic Projections for the P...


### Searching for individual series: oil prices and PCE inflation
When we do not know the release, we can search series by keywords. Results are ordered by popularity.

In [11]:
# Search oil price series. The monthly WTI price (MCOILWTICO) will be selected
res = pd.DataFrame(fred_get('series/search', search_text='crude oil WTI',
                            order_by='popularity', limit=10)['seriess'])
res[['id', 'title', 'frequency', 'units', 'seasonal_adjustment']]

,id,title,frequency,units,seasonal_adjustment
0,DCOILWTICO,Crude Oil Prices: West Texas Intermediate (WTI...,Daily,Dollars per Barrel,Not Seasonally Adjusted
1,WTISPLC,Spot Crude Oil Price: West Texas Intermediate ...,Monthly,Dollars per Barrel,Not Seasonally Adjusted
2,MCOILWTICO,Crude Oil Prices: West Texas Intermediate (WTI...,Monthly,Dollars per Barrel,Not Seasonally Adjusted
3,POILWTIUSDM,Global price of WTI Crude,Monthly,U.S. Dollars per Barrel,Not Seasonally Adjusted
4,WCOILWTICO,Crude Oil Prices: West Texas Intermediate (WTI...,"Weekly, Ending Friday",Dollars per Barrel,Not Seasonally Adjusted
5,POILWTIUSDQ,Global price of WTI Crude,Quarterly,U.S. Dollars per Barrel,Not Seasonally Adjusted
6,ACOILWTICO,Crude Oil Prices: West Texas Intermediate (WTI...,Annual,Dollars per Barrel,Not Seasonally Adjusted
7,POILWTIUSDA,Global price of WTI Crude,Annual,U.S. Dollars per Barrel,Not Seasonally Adjusted
8,NASDAQQUSOITR,Credit Suisse NASDAQ WTI Crude Oil FLOWS 106 T...,Daily,Index,Not Seasonally Adjusted
9,NASDAQQUSOI,Credit Suisse NASDAQ WTI Crude Oil FLOWS 106 P...,Daily,Index,Not Seasonally Adjusted


In [12]:
# Search PCE price index series. The codes PCEPI (headline) and PCEPILFE (core) will be selected
res = pd.DataFrame(fred_get('series/search', search_text='personal consumption expenditures price index',
                            order_by='popularity', limit=10)['seriess'])
res[['id', 'title', 'frequency', 'units', 'seasonal_adjustment']]

,id,title,frequency,units,seasonal_adjustment
0,PCEPI,Personal Consumption Expenditures: Chain-type ...,Monthly,Index 2017=100,Seasonally Adjusted
1,PCEPILFE,Personal Consumption Expenditures Excluding Fo...,Monthly,Index 2017=100,Seasonally Adjusted
2,DPCCRV1Q225SBEA,Personal Consumption Expenditures (PCE) Exclud...,Quarterly,Percent Change from Preceding Period,Seasonally Adjusted Annual Rate
3,PCETRIM12M159SFRBDAL,Trimmed Mean PCE Inflation Rate,Monthly,Percent Change from Year Ago,Seasonally Adjusted
4,PCETRIM1M158SFRBDAL,Trimmed Mean PCE Inflation Rate,Monthly,Percent Change at Annual Rate,Seasonally Adjusted
5,DFUNRG3A086NBEA,Personal consumption expenditures: Funeral and...,Annual,Index 2017=100,Not Seasonally Adjusted
6,PCECTPI,Personal Consumption Expenditures: Chain-type ...,Quarterly,Index 2017=100,Seasonally Adjusted
7,IA001260M,Personal Consumption Expenditures: Services Ex...,Monthly,Index 2017=100,Seasonally Adjusted
8,DPCERD3Q086SBEA,Personal consumption expenditures (implicit pr...,Quarterly,Index 2017=100,Seasonally Adjusted
9,BPCCRO1Q156NBEA,Personal Consumption Expenditures Excluding Fo...,Quarterly,Percent Change from Quarter One Year Ago,Seasonally Adjusted


### Getting the metadata of a series

In [13]:
# Metadata of the PCE price index: units, frequency, seasonal adjustment and notes
fred_get('series', series_id='PCEPI')['seriess'][0]

{'id': 'PCEPI',
 'realtime_start': '2026-08-26',
 'realtime_end': '2026-08-26',
 'title': 'Personal Consumption Expenditures: Chain-type Price Index',
 'observation_start': '1959-01-01',
 'observation_end': '2026-07-01',
 'frequency': 'Monthly',
 'frequency_short': 'M',
 'units': 'Index 2017=100',
 'units_short': 'Index 2017=100',
 'seasonal_adjustment': 'Seasonally Adjusted',
 'seasonal_adjustment_short': 'SA',
 'last_updated': '2026-08-26 07:43:48-05',
 'popularity': 82,
 'notes': "BEA Account Code: DPCERG\r\n\r\nThe Personal Consumption Expenditures Price Index is a measure of the prices that people living in the United States, or those buying on their behalf, pay for goods and services. The change in the PCE price index is known for capturing inflation (or deflation) across a wide range of consumer expenses and reflecting changes in consumer behavior. For example, if the price of beef rises, shoppers may buy less beef and more chicken. \r\n\r\nThe PCE Price Index is produced by the

In [14]:
# Release to which a series belongs
fred_get('series/release', series_id='MCOILWTICO')['releases'][0]

{'id': 212,
 'realtime_start': '2026-09-22',
 'realtime_end': '2026-09-22',
 'name': 'Spot Prices',
 'press_release': False,
 'link': 'https://www.eia.gov/dnav/pet/pet_pri_spt_s1_d.htm'}

These codes (release ID 326 and series IDs) will be necessary to construct the requests in Part 2.